# 03 — Inferência no teste

Execução final: **1.147 perguntas × 6 representações × modelos congelados**, com a configuração gravada pelo notebook 02 em `frozen_config.json`. Este notebook **não decide nada**: só confere que o prompt é o congelado, gera as respostas e grava.

- Decodificação gulosa, `max_new_tokens` e precisão vêm do congelamento (seção 3 da metodologia).
- Um arquivo JSONL por modelo × condição em `test/`, gravado a cada lote na pasta `NLP - Checkpoints/results` do Google Drive (`F.DRIVE_FOLDER`). **Se a sessão cair, abra o notebook e rode tudo de novo**: o que já foi feito é pulado, um modelo completo nem é carregado e os lotes restantes têm a mesma composição de antes. Perde-se, no máximo, o lote que estava em andamento.
- A acurácia **não** é mostrada aqui. A análise é feita uma vez, no notebook 04, com todos os resultados.

**Antes de rodar no Colab:** runtime com GPU compatível com a precisão congelada (bf16: 22 GB ou mais) e o repositório no GitHub no mesmo estado em que o piloto foi rodado. Os dois modelos são abertos: não é preciso token do Hugging Face.

In [ ]:
# Ambiente: repositório, módulo compartilhado e pasta de resultados
import os, sys, subprocess, pathlib, json, platform, datetime, gc

REPO_URL = "https://github.com/Arthur-Marques-IA/NLP_Fin-RAG.git"

def find_repo():
    here = pathlib.Path.cwd()
    for p in (here, here.parent, here / "NLP_Fin-RAG"):
        if (p / "src" / "finrag.py").exists():
            return p
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "NLP_Fin-RAG"], check=True)
    return here / "NLP_Fin-RAG"

REPO = find_repo()
os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))
import finrag as F
import pandas as pd

try:
    from google.colab import drive, userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if os.environ.get("FINRAG_RESULTS_DIR"):
    RESULTS_DIR = pathlib.Path(os.environ["FINRAG_RESULTS_DIR"])
elif IN_COLAB:
    drive.mount("/content/drive")
    RESULTS_DIR = F.drive_results_dir(os.environ.get("FINRAG_DRIVE_FOLDER", F.DRIVE_FOLDER))
else:
    RESULTS_DIR = REPO / "results"
DATA_DIR = REPO / "data"
FROZEN_PATH = RESULTS_DIR / "frozen_config.json"

try:
    HF_TOKEN = userdata.get("HF_TOKEN") if IN_COLAB else os.environ.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

import torch, transformers
HAS_GPU = torch.cuda.is_available()
GPU_NAME = torch.cuda.get_device_properties(0).name if HAS_GPU else None
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if HAS_GPU else 0.0
print("Repositório:", REPO, "| commit:", F.git_sha(REPO))
print("Resultados em:", RESULTS_DIR)
print("GPU:", f"{GPU_NAME} ({GPU_GB:.1f} GB)" if HAS_GPU else "nenhuma")
print("torch", torch.__version__, "| transformers", transformers.__version__)

## Configuração

`MODELS_TO_RUN = None` roda todos os modelos congelados. Para dividir o trabalho entre sessões (ou entre contas do Colab que escrevem na mesma pasta do Drive), escolha um subconjunto, por exemplo `["phi4-mini"]`.

In [ ]:
MODELS_TO_RUN = None
REPS_TO_RUN = list(F.REPRESENTATIONS)
SAVE_EVERY = 1                     # grava no Drive a cada N lotes (1 = a cada 4 perguntas)

## 1. Conferência do congelamento

O teste só roda se o prompt atual for **exatamente** o que passou no piloto: mesmo template, mesmos exemplos, mesmo *k* e os mesmos 6.882 prompts (o hash cobre também a serialização das tabelas e os dados).

In [ ]:
assert FROZEN_PATH.exists(), "Não há frozen_config.json: rode o notebook 02 até a seção 8."
frozen = F.load_json(FROZEN_PATH)

train = F.load_finqa("train", DATA_DIR)
test = F.load_finqa("test", DATA_DIR)
shots = F.get_fewshot(train)
K = frozen["k"]

checks = {
    "hash do template": (F.prompt_hash(K), frozen["prompt_hash"]),
    "hash dos prompts do teste": (F.prompts_fingerprint(test, shots, K), frozen["prompts_fingerprint_test"]),
    "exemplos few-shot": (list(F.FEWSHOT_IDS), frozen["fewshot_ids"]),
    "k": (F.K_FROZEN, K),
    "max_new_tokens": (F.MAX_NEW_TOKENS, frozen["max_new_tokens"]),
}
for name, (now, then) in checks.items():
    print(f"{'OK  ' if now == then else 'ERRO'} {name}: {now}")
assert all(now == then for now, then in checks.values()), \
    "O código mudou depois do congelamento. Volte ao commit do piloto ou refaça o piloto (notebook 02)."

PRECISION, BATCH_SIZE = frozen["precision"], frozen["batch_size"]
outside = sorted(set(frozen["models"]) - set(F.MODELS))
assert not outside, f"O congelamento inclui modelos fora do estudo: {outside}. Refaça o congelamento (notebook 02)."
MODELS = [m for m in F.MODELS if m in frozen["models"] and (MODELS_TO_RUN is None or m in MODELS_TO_RUN)]   # ordem de execução
print(f"\nPrecisão: {PRECISION} | lote: {BATCH_SIZE} | modelos: {MODELS}")
print(f"Congelado em {frozen['frozen_at']} (commit {frozen['git_sha']})")
if frozen.get("git_dirty") or F.git_dirty(REPO):
    print("ATENÇÃO: o congelamento ou esta sessão usa código modificado sem commit. Resultados assim não entram no estudo.")
if frozen["git_sha"] != F.git_sha(REPO):
    print("Aviso: o commit atual é diferente do commit do congelamento (os hashes acima conferem, então o prompt é o mesmo).")

assert HAS_GPU, "A inferência precisa de GPU."
assert PRECISION != "bf16" or GPU_GB >= 22, \
    f"O congelamento usa bf16, que não cabe nesta GPU ({GPU_GB:.0f} GB). Use uma L4/A100."
if PRECISION == "4bit":
    try:
        import bitsandbytes
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes", "accelerate"], check=True)

## 2. Andamento

In [ ]:
def progress():
    """Perguntas já gravadas por modelo × condição."""
    return F.progress(RESULTS_DIR, "test", list(frozen["models"]), len(test))

status = progress()
print(status.to_string())
print(f"\nFeitas: {int(status.values.sum())} de {len(test) * status.size} gerações")

## 3. Geração

Um modelo por vez; dentro de cada modelo, uma condição por vez. Cada sessão grava um arquivo próprio em `test/sessions/` (GPU, versões, commit), como pede a seção 9 da metodologia.

Antes de gerar, cada modelo passa por um **teste de memória**: o prompt mais longo de cada condição pendente é processado sozinho. Se ele não couber na GPU, a sessão para aqui, no começo, e não horas depois.

In [ ]:
SESSION_ID = datetime.datetime.now().strftime("%Y%m%dT%H%M%S") + "_" + "-".join(MODELS)
F.save_json(RESULTS_DIR / "test" / "sessions" / f"{SESSION_ID}.json", {
    "id": SESSION_ID,
    "gpu": GPU_NAME, "gpu_gb": round(GPU_GB, 1), "precision": PRECISION, "batch_size": BATCH_SIZE,
    "python": platform.python_version(), "torch": torch.__version__, "transformers": transformers.__version__,
    "git_sha": F.git_sha(REPO), "git_dirty": F.git_dirty(REPO),
    "prompt_hash": frozen["prompt_hash"], "models": MODELS, "reps": REPS_TO_RUN,
})

for name in MODELS:
    pending = [rep for rep in REPS_TO_RUN
               if len(F.read_jsonl(F.result_path(RESULTS_DIR, "test", name, rep))) < len(test)]
    if not pending:
        print(f"{name}: completo, nada a fazer.")
        continue
    tok, model, revision = F.load_model(name, PRECISION, HF_TOKEN, frozen["models"][name]["revision"])
    assert frozen["models"][name]["revision"] in (None, revision), (name, revision)
    for rep in pending:                               # teste de memória: o prompt mais longo de cada condição
        longest = max((F.build_prompt(ex, rep, shots, K) for ex in test),
                      key=lambda p: F.count_tokens(tok, F.chat_text(tok, name, p)))
        F._generate_safe(tok, model, name, [longest], 1)
    print(f"{name}: o prompt mais longo de cada condição pendente cabe na memória.")
    for rep in pending:
        total_done = int(progress().values.sum())
        total_all = len(test) * len(F.REPRESENTATIONS) * len(frozen["models"])
        print(f"Geral: {total_done} de {total_all} gerações gravadas ({total_done / total_all:.1%})")
        F.run_condition(tok, model, name, test, rep, shots,
                        F.result_path(RESULTS_DIR, "test", name, rep),
                        split="test", precision=PRECISION, k=K, batch_size=BATCH_SIZE,
                        save_every=SAVE_EVERY)
    del model, tok
    gc.collect()
    torch.cuda.empty_cache()

## 4. Conferência final

Verifica só a **integridade** do que foi gravado: quantidade de perguntas, prompt e precisão únicos, falhas de formato e saídas cortadas. A acurácia fica para o notebook 04.

In [ ]:
status = progress()
print(status.to_string())
missing = int(len(test) * status.size - status.values.sum())
print("\nTudo completo." if missing == 0 else f"\nFaltam {missing} gerações: rode a seção 3 de novo.")

res = F.load_results(RESULTS_DIR, "test")
if len(res):
    assert set(res["prompt_hash"]) == {frozen["prompt_hash"]} and set(res["precision"]) == {PRECISION}
    print("\nDesfecho das saídas (fração por modelo × condição; só 'resposta' pode ser acerto):\n")
    print(F.outcome_rates(res, test).round(3).to_string())
    print("\nRegistros gerados em lote menor que o padrão (último lote de cada condição ou falta de memória):",
          int((res["batch_size"] < BATCH_SIZE).sum()))